# Medical RAG Baseline (MS MARCO QA v1.1)

1차 목표의 baseline 파이프라인: **의료 근거 문서를 검색하고, 그 근거로만 답하며, 근거가 없으면 답하지 않는다.**

```
Query → BM25 / Dense(FAISS) → Hybrid(RRF) → Cross-Encoder Rerank → 임계값 게이트 → LLM(근거 전용 프롬프트) → Answer
```

| 단계 | 내용 |
|---|---|
| 1. 데이터 | v1.1 train+validation passage를 펼치고, 정규화와 중복 제거 후 **의료 URL 화이트리스트**로 코퍼스 구성 (약 6만 개) |
| 2. 평가 세트 | 의료 질문 = 정답이 있고 `is_selected` passage가 화이트리스트 도메인인 질문 / 도메인 밖 질문 = 의료 passage가 전혀 없고 의료 키워드도 없는 질문 |
| 3. 검색 | BM25(bm25s) / Dense(bge-base + FAISS) / Hybrid(RRF) / +Reranker 비교 (Recall@10, MRR@10) |
| 4. 답변 불가 판단 | rerank 최고 점수 임계값을 **train 튜닝 세트**에서 정하고 validation에서 평가 |
| 5. 생성 | Qwen2.5-7B-Instruct, 문장형 답변 (ROUGE-L, 정답 포함률) |

**런타임**: Colab Pro, GPU **A100 또는 L4**. 7B 모델을 bf16으로 올리면 약 15GB를 사용합니다.

| 세트 | 출처 | 용도 |
|---|---|---|
| `med_test` | validation 의료 질문 전체 (약 840개) | 검색, 생성, 오거절률 평가 |
| `ood_test` | validation 도메인 밖 질문 1,000개 | 거절률 평가 |
| `med_tune` / `ood_tune` | train에서 각 500개 | 임계값 튜닝 (test와 분리) |

## 0. 환경 설정

In [ ]:
%pip install -q -U datasets bm25s PyStemmer faiss-cpu sentence-transformers rouge-score accelerate

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, re, json, random
from pathlib import Path
from collections import defaultdict
from urllib.parse import urlparse

import numpy as np
import pandas as pd

# ===== 경로 =====
BASE_DIR = Path(os.environ.get("RAG_BASE_DIR", "/content/drive/MyDrive/RAG-Chat"))
DATA_DIR = BASE_DIR / "data" / "processed"
INDEX_DIR = BASE_DIR / "index"
RESULT_DIR = BASE_DIR / "results" / "baseline"
for d in (DATA_DIR, INDEX_DIR, RESULT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ===== 실험 설정 =====
SEED = 42
DATASET_NAME, DATASET_VERSION = "microsoft/ms_marco", "v1.1"

N_OOD_TEST = 1000      # validation 도메인 밖 질문 수
N_TUNE = 500           # train 튜닝 세트 (의료 / 도메인 밖 각각)

EMBED_MODEL = "BAAI/bge-base-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "  # bge 검색용 쿼리 instruction
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"

K_CAND = 100           # BM25 / Dense 후보 수
K_RERANK = 50          # Hybrid 상위 몇 개를 rerank할지
K_EVAL = 10            # Recall@10, MRR@10
K_CTX = 5              # LLM에 넣을 passage 수
RRF_K = 60
MAX_NEW_TOKENS = 128
GEN_BATCH = 16

REFUSAL = "I cannot answer this question based on the provided documents."

random.seed(SEED)
np.random.seed(SEED)

## 1. 데이터 준비

### 1-1. 의료 URL 화이트리스트
v1.1의 전체 도메인 분포를 보고 **의료 정보 제공이 주목적인 사이트**만 골랐습니다.
`host == 도메인`이거나 `*.도메인`인 경우를 포함합니다. 예를 들어 `nih.gov`는 `ncbi.nlm.nih.gov`, `ghr.nlm.nih.gov`, `nhlbi.nih.gov`를 포함합니다.

일부러 제외한 사이트:
- 이름은 health/med여도 의료 지식이 아닌 사이트: `healthcare-salaries.com`(연봉), `health.costhelper.com`(비용), `healthgrades.com`(의사 정보), `medicare.gov`(제도), 아기 이름 사이트, 수의학 사이트
- 의료 인접(B): `livestrong.com`, `healthyeating.sfgate.com`. 확장할 때 추가를 검토합니다.
- 범용 사이트: `en.wikipedia.org`. 의료 문서가 섞여 있지만 URL만으로는 구분할 수 없습니다.

In [ ]:
MEDICAL_DOMAINS = [
    "webmd.com", "webmd.boots.com", "mayoclinic.org", "mayoclinic.com", "medicinenet.com",
    "healthline.com", "medical-dictionary.thefreedictionary.com", "nih.gov", "medlineplus.gov",
    "cdc.gov", "everydayhealth.com", "nhs.uk", "medicalnewstoday.com", "rxlist.com",
    "medscape.com", "emedicinehealth.com", "wisegeekhealth.com", "health.howstuffworks.com",
    "cancer.org", "clevelandclinic.org", "patient.info", "hopkinsmedicine.org", "healthtap.com",
    "news-medical.net", "cancerresearchuk.org", "americanpregnancy.org", "spine-health.com",
    "who.int", "cancer.gov", "womenshealth.gov", "md-health.com", "heart.org", "healthhype.com",
    "familydoctor.org", "goodrx.com", "medhelp.org", "symptomfind.com", "medilexicon.com",
    "psychcentral.com", "healthcommunities.com", "med-health.net", "diabetes.org",
    "themedicalquestions.com", "health.harvard.edu", "drugs.com", "cancercenter.com",
    "rightdiagnosis.com", "innerbody.com", "labtestsonline.org", "sharecare.com", "umm.edu",
    "kidshealth.org", "health.com", "healthguideinfo.com", "medicalhealthtests.com",
]

def get_host(url):
    host = (urlparse(url).hostname or "").lower()
    return host[4:] if host.startswith("www.") else host

def is_medical_host(host):
    return any(host == d or host.endswith("." + d) for d in MEDICAL_DOMAINS)

def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def has_answer(answers):
    return any(
        str(a).strip() and str(a).strip().lower().rstrip(".") != "no answer present"
        for a in (answers if answers is not None else [])
    )

### 1-2. passage 펼치기와 코퍼스 생성
- 한 행이 (질문, passage) 하나가 되도록 펼친 뒤 공백을 정규화하고 빈 passage를 제거합니다.
- 같은 텍스트는 하나의 `pid`로 합칩니다. 이 `pid`로 평가 정답과 검색 결과를 연결합니다.
- 코퍼스에는 의료 도메인 passage를 **정답 여부와 관계없이 모두** 넣습니다. 그래야 정답이 아닌 의료 문서가 검색 경쟁 상대가 되어 Retriever 평가가 의미를 가집니다.

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME, name=DATASET_VERSION)
qa = {split: dataset[split].to_pandas() for split in ("train", "validation")}
for split, df in qa.items():
    print(split, len(df))

In [ ]:
rows = []
for split, df in qa.items():
    for qid, p in zip(df["query_id"], df["passages"]):
        for text, sel, url in zip(p["passage_text"], p["is_selected"], p["url"]):
            rows.append((split, int(qid), normalize_text(text), int(sel), url))

flat = pd.DataFrame(rows, columns=["split", "query_id", "text", "is_selected", "url"])
flat = flat[flat["text"] != ""]
flat["host"] = flat["url"].map(get_host)
host_is_med = {h: is_medical_host(h) for h in flat["host"].unique()}
flat["is_medical"] = flat["host"].map(host_is_med)

corpus = (
    flat[flat["is_medical"]]
    .drop_duplicates("text")[["text", "url", "host"]]
    .reset_index(drop=True)
)
corpus.insert(0, "pid", np.arange(len(corpus)))   # pid == 행 번호 (인덱스 위치와 동일)
text2pid = dict(zip(corpus["text"], corpus["pid"]))

print(f"전체 passage: {len(flat):,} / 고유 텍스트: {flat['text'].nunique():,}")
print(f"의료 코퍼스: {len(corpus):,}")
print(corpus["host"].value_counts().head(10).to_string())

### 1-3. 평가 세트
- **의료 질문**: 정답이 있고, `is_selected=1`인 passage 중 하나 이상이 의료 도메인인 질문입니다. 그 passage들의 `pid`가 검색 정답(`gold_pids`)이 됩니다.
- **도메인 밖 질문**: 정답이 있지만 의료 도메인 passage가 하나도 없고, `baseline.ipynb`의 의료 키워드에도 걸리지 않는 질문입니다. 코퍼스에 근거가 없으므로 **거절해야 정답**입니다.

알려진 한계: URL 기준이라 `what age do you give puppies parvo shots`처럼 의료가 아닌 질문이 일부 섞입니다. 질문 분류(A/B/C/U) 결과가 나오면 교체할 예정입니다.

In [ ]:
MEDICAL_KEYWORDS = [
    "disease", "disorder", "syndrome", "diabetes", "cancer", "asthma", "pneumonia", "infection",
    "arthritis", "stroke", "hypertension", "anemia", "symptom", "pain", "fever", "headache",
    "nausea", "vomiting", "dizziness", "fatigue", "swelling", "bleeding", "diagnosis", "diagnose",
    "blood test", "biopsy", "mri", "ct scan", "x-ray", "ultrasound", "medication", "medicine",
    "drug", "antibiotic", "insulin", "vaccine", "treatment", "therapy", "surgery", "prescription",
    "side effect", "heart", "brain", "kidney", "liver", "lung", "blood pressure", "hormone",
    "immune system", "pregnancy", "depression", "anxiety", "adhd", "mental health", "bipolar",
]
keyword_pattern = re.compile(
    r"(?<!\w)(?:" + "|".join(re.escape(k) for k in MEDICAL_KEYWORDS) + r")(?!\w)",
    flags=re.IGNORECASE,
)

def build_query_sets(split):
    df = qa[split]
    fs = flat[flat["split"] == split]
    gold = (
        fs[fs["is_medical"] & (fs["is_selected"] == 1)]
        .groupby("query_id")["text"]
        .apply(lambda ts: sorted({text2pid[t] for t in ts}))
    )
    any_med = set(fs.loc[fs["is_medical"], "query_id"])

    base = pd.DataFrame({
        "query_id": df["query_id"].astype(int),
        "query": df["query"].map(normalize_text),
        "query_type": df["query_type"],
        "answers": df["answers"].map(list),
    })
    answerable = df["answers"].map(has_answer)

    med = base[answerable & base["query_id"].isin(gold.index)].copy()
    med["gold_pids"] = med["query_id"].map(gold)

    ood = base[
        answerable
        & ~base["query_id"].isin(any_med)
        & ~base["query"].str.contains(keyword_pattern)
    ].copy()
    ood["gold_pids"] = [[] for _ in range(len(ood))]
    return med.reset_index(drop=True), ood.reset_index(drop=True)

med_val, ood_val = build_query_sets("validation")
med_train, ood_train = build_query_sets("train")

qsets = {
    "med_test": med_val,
    "ood_test": ood_val.sample(N_OOD_TEST, random_state=SEED).reset_index(drop=True),
    "med_tune": med_train.sample(N_TUNE, random_state=SEED).reset_index(drop=True),
    "ood_tune": ood_train.sample(N_TUNE, random_state=SEED).reset_index(drop=True),
}
for name, q in qsets.items():
    print(f"{name:9s} {len(q):5d}")
display(qsets["med_test"].head())

In [ ]:
corpus.to_parquet(DATA_DIR / "medical_corpus.parquet", index=False)
for name, q in qsets.items():
    q.to_parquet(DATA_DIR / f"{name}.parquet", index=False)
print("saved to", DATA_DIR)

## 2. 검색 (Retrieval)

`pid`는 코퍼스의 행 번호이므로 BM25와 FAISS가 돌려주는 인덱스가 곧 `pid`입니다.

### 2-1. BM25 (bm25s, 영어 불용어와 Snowball stemmer)

In [ ]:
import bm25s, Stemmer

stemmer = Stemmer.Stemmer("english")
passages = corpus["text"].tolist()

bm25 = bm25s.BM25()
bm25.index(bm25s.tokenize(passages, stopwords="en", stemmer=stemmer))

def bm25_search(queries, k=K_CAND):
    tokens = bm25s.tokenize(queries, stopwords="en", stemmer=stemmer, show_progress=False)
    ids, scores = bm25.retrieve(tokens, k=k, show_progress=False)
    return ids, scores

### 2-2. Dense (bge-base-en-v1.5 + FAISS)
임베딩은 Drive에 캐시합니다. 코퍼스가 바뀌면 캐시 파일을 지우고 다시 계산하세요.

In [ ]:
import torch, faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL, device="cuda")
embedder.half()

emb_path = INDEX_DIR / f"{EMBED_MODEL.split('/')[-1]}_{len(corpus)}.npy"
if emb_path.exists():
    corpus_emb = np.load(emb_path)
else:
    corpus_emb = embedder.encode(
        passages, batch_size=256, normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=True,
    ).astype("float32")
    np.save(emb_path, corpus_emb)

faiss_index = faiss.IndexFlatIP(corpus_emb.shape[1])   # 정규화된 벡터의 내적 = 코사인 유사도
faiss_index.add(corpus_emb)

def dense_search(queries, k=K_CAND):
    q_emb = embedder.encode(
        [QUERY_PREFIX + q for q in queries], batch_size=256,
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype("float32")
    scores, ids = faiss_index.search(q_emb, k)
    return ids, scores

print(corpus_emb.shape)

### 2-3. Hybrid (RRF)과 Cross-Encoder Rerank

In [ ]:
def rrf_fuse(*id_lists_per_run, k=RRF_K, top=K_RERANK):
    """run마다 (n_query, K) 형태의 id 배열을 받아 질의별로 RRF로 합친 상위 top개 pid 리스트를 반환."""
    fused = []
    for per_query in zip(*id_lists_per_run):
        score = defaultdict(float)
        for ids in per_query:
            for rank, pid in enumerate(ids):
                score[int(pid)] += 1.0 / (k + rank + 1)
        fused.append(sorted(score, key=score.get, reverse=True)[:top])
    return fused

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(RERANK_MODEL, device="cuda", max_length=512)

def rerank(queries, candidates):
    """candidates: 질의별 pid 리스트 → (정렬된 pid 리스트, 정렬된 점수 리스트)"""
    pairs = [(q, passages[pid]) for q, cand in zip(queries, candidates) for pid in cand]
    flat_scores = reranker.predict(pairs, batch_size=256, show_progress_bar=False)
    out_ids, out_scores, pos = [], [], 0
    for cand in candidates:
        s = np.asarray(flat_scores[pos:pos + len(cand)])
        pos += len(cand)
        order = np.argsort(-s)
        out_ids.append([cand[i] for i in order])
        out_scores.append(s[order].tolist())
    return out_ids, out_scores

### 2-4. 모든 질의 세트 검색 실행

In [ ]:
runs = {}   # runs[qset][method] = 질의별 pid 리스트 (순위순)
rerank_scores = {}
for name, q in qsets.items():
    queries = q["query"].tolist()
    bm25_ids, _ = bm25_search(queries)
    dense_ids, _ = dense_search(queries)
    hybrid_ids = rrf_fuse(bm25_ids, dense_ids)
    rr_ids, rr_scores = rerank(queries, hybrid_ids)
    runs[name] = {
        "BM25": [list(map(int, x)) for x in bm25_ids],
        "Dense": [list(map(int, x)) for x in dense_ids],
        "Hybrid": hybrid_ids,
        "Hybrid+Rerank": rr_ids,
    }
    rerank_scores[name] = rr_scores
    print(f"{name} done")

### 2-5. 검색 평가: Recall@10, MRR@10 (`med_test`)
- Recall@10: 정답 passage 중 상위 10개 안에 들어온 비율의 평균
- MRR@10: 첫 정답 순위의 역수 평균 (10위 밖이면 0)

In [ ]:
def retrieval_metrics(run, gold, k=K_EVAL):
    recall, mrr = [], []
    for ids, g in zip(run, gold):
        g = set(g)
        top = list(ids)[:k]
        hits = [r for r, pid in enumerate(top) if pid in g]
        recall.append(len(set(top) & g) / len(g))
        mrr.append(1.0 / (hits[0] + 1) if hits else 0.0)
    return {f"Recall@{k}": np.mean(recall), f"MRR@{k}": np.mean(mrr)}

gold = qsets["med_test"]["gold_pids"].tolist()
retrieval_table = pd.DataFrame(
    {method: retrieval_metrics(run, gold) for method, run in runs["med_test"].items()}
).T.round(4)
retrieval_table.to_csv(RESULT_DIR / "retrieval_metrics.csv")
retrieval_table

> 참고: `ms-marco-MiniLM` reranker는 MS MARCO passage ranking으로 학습된 모델이라 이 데이터에서는 점수가 실제보다 좋게 나올 수 있습니다. 비교용으로 `BAAI/bge-reranker-base`도 돌려보는 것을 권장합니다.

## 3. 답변 불가 판단: rerank 최고 점수 임계값

rerank 1위 점수가 임계값보다 낮으면 LLM을 호출하지 않고 거절합니다.
임계값은 **train 튜닝 세트**(`med_tune` 500개, `ood_tune` 500개)에서 balanced accuracy가 최대가 되도록 정하고, test 세트에는 그대로 적용합니다.

In [ ]:
def top_scores(name):
    return np.array([s[0] for s in rerank_scores[name]])

tune_med, tune_ood = top_scores("med_tune"), top_scores("ood_tune")
candidates = np.unique(np.concatenate([tune_med, tune_ood]))
bal_acc = [((tune_med >= t).mean() + (tune_ood < t).mean()) / 2 for t in candidates]
THRESHOLD = float(candidates[int(np.argmax(bal_acc))])
print(f"THRESHOLD = {THRESHOLD:.4f}  (tune balanced acc = {max(bal_acc):.4f})")

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.hist(tune_med, bins=50, alpha=0.6, label="med_tune")
plt.hist(tune_ood, bins=50, alpha=0.6, label="ood_tune")
plt.axvline(THRESHOLD, color="k", linestyle="--", label="threshold")
plt.xlabel("top-1 rerank score"); plt.ylabel("count"); plt.legend(); plt.tight_layout(); plt.show()

## 4. 답변 생성

- **근거 전용 프롬프트**: passage에 없는 내용은 쓰지 않고, 답이 없으면 정해진 거절 문장을 그대로 출력하게 합니다.
- **문장형 답변**: 질문의 주어를 살린 1~2문장으로 답하게 합니다.
- `No-Retrieval`은 passage 없이 모델 지식만으로 답한 결과입니다. 검색 효과를 보여주기 위한 **비교 기준**일 뿐 실제 파이프라인에서는 쓰지 않습니다.

In [ ]:
SYSTEM_RAG = f"""You are a medical information assistant. Answer the question using ONLY the information in the provided passages.
Rules:
1. Do not use any knowledge that is not stated in the passages.
2. Answer in one or two complete sentences that restate the subject of the question.
3. If the passages do not contain the answer, reply exactly: {REFUSAL}"""

SYSTEM_CLOSED = "You are a medical information assistant. Answer the question in one or two complete sentences."

def build_messages(query, pids=None):
    if pids is None:
        return [{"role": "system", "content": SYSTEM_CLOSED},
                {"role": "user", "content": f"Question: {query}\nAnswer:"}]
    ctx = "\n\n".join(f"[{i}] {passages[pid]}" for i, pid in enumerate(pids, 1))
    return [{"role": "system", "content": SYSTEM_RAG},
            {"role": "user", "content": f"Passages:\n{ctx}\n\nQuestion: {query}\nAnswer:"}]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL, padding_side="left")
llm = AutoModelForCausalLM.from_pretrained(GEN_MODEL, torch_dtype=torch.bfloat16, device_map="cuda")
llm.eval()

@torch.no_grad()
def generate(messages_list, batch_size=GEN_BATCH):
    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
               for m in messages_list]
    order = np.argsort([len(p) for p in prompts])          # 길이순 배치로 padding 낭비 줄이기
    outputs = [None] * len(prompts)
    for i in tqdm(range(0, len(prompts), batch_size)):
        idx = order[i:i + batch_size]
        enc = tokenizer([prompts[j] for j in idx], return_tensors="pt", padding=True).to(llm.device)
        out = llm.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                           temperature=None, top_p=None, top_k=None)
        texts = tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        for j, t in zip(idx, texts):
            outputs[j] = t.strip()
    return outputs

### 4-1. 생성 평가 지표
- **ROUGE-L (F1)**: 정답(`answers`)이 여러 개면 최댓값을 씁니다.
- **정답 포함률 (answer recall)**: 정답 토큰 중 생성 답변에 포함된 비율입니다. 문장형 답변은 짧은 정답보다 길어서 ROUGE 정밀도가 낮게 나오므로, 정답 내용을 담았는지 이 지표로 봅니다.
- **거절률**: 답변에 거절 문장이 들어 있는 비율입니다.

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def _tokens(s):
    return re.findall(r"[a-z0-9]+", s.lower())

def is_refusal(pred):
    return "cannot answer" in pred.lower()

def answer_recall(pred, ref):
    ref_t, pred_t = _tokens(ref), set(_tokens(pred))
    return sum(t in pred_t for t in ref_t) / len(ref_t) if ref_t else 0.0

def generation_metrics(preds, answers_list):
    rouge = [max(scorer.score(a, p)["rougeL"].fmeasure for a in ans) for p, ans in zip(preds, answers_list)]
    recall = [max(answer_recall(p, a) for a in ans) for p, ans in zip(preds, answers_list)]
    return {"ROUGE-L": np.mean(rouge), "AnswerRecall": np.mean(recall),
            "RefusalRate": np.mean([is_refusal(p) for p in preds])}

### 4-2. 검색 방식별 생성 비교 (`med_test`, 게이트 없이 상위 5개 passage 사용)
7B 모델 기준으로 A100에서 설정 하나당 약 5분이 걸립니다. 시간이 부족하면 `GEN_CONFIGS`를 줄이세요.

In [ ]:
GEN_CONFIGS = ["No-Retrieval", "BM25", "Dense", "Hybrid", "Hybrid+Rerank"]

mt = qsets["med_test"]
preds = {}
for cfg in GEN_CONFIGS:
    if cfg == "No-Retrieval":
        msgs = [build_messages(q) for q in mt["query"]]
    else:
        msgs = [build_messages(q, ids[:K_CTX]) for q, ids in zip(mt["query"], runs["med_test"][cfg])]
    preds[cfg] = generate(msgs)

gen_table = pd.DataFrame(
    {cfg: generation_metrics(p, mt["answers"].tolist()) for cfg, p in preds.items()}
).T.round(4)
gen_table.to_csv(RESULT_DIR / "generation_metrics.csv")
gen_table

## 5. 최종 파이프라인 평가 (Hybrid+Rerank+게이트)

| 지표 | 의미 |
|---|---|
| OOD 거절률 | 도메인 밖 질문 중 거절한 비율 (게이트 또는 LLM 거절, 높을수록 좋음) |
| 의료 오거절률 | 의료 질문인데 거절한 비율 (낮을수록 좋음) |
| 게이트 거절 / LLM 거절 | 어느 단계에서 거절했는지 |

In [ ]:
def run_pipeline(name, reuse_preds=None):
    q = qsets[name]
    passed = top_scores(name) >= THRESHOLD
    answers = [REFUSAL] * len(q)
    todo = np.where(passed)[0]
    if reuse_preds is not None:          # med_test는 4-2에서 생성한 Hybrid+Rerank 결과를 재사용
        for i in todo:
            answers[i] = reuse_preds[i]
    else:
        msgs = [build_messages(q["query"][i], runs[name]["Hybrid+Rerank"][i][:K_CTX]) for i in todo]
        for i, a in zip(todo, generate(msgs)):
            answers[i] = a
    refused = np.array([is_refusal(a) for a in answers])
    return answers, {
        "RefusalRate": refused.mean(),
        "GateRefusal": (~passed).mean(),
        "LLMRefusal": (refused & passed).mean(),
    }

final_med, med_stats = run_pipeline("med_test", reuse_preds=preds["Hybrid+Rerank"])
final_ood, ood_stats = run_pipeline("ood_test")

final_table = pd.DataFrame({"med_test (오거절)": med_stats, "ood_test (거절)": ood_stats}).T.round(4)
final_table["ROUGE-L"] = [generation_metrics(final_med, mt["answers"].tolist())["ROUGE-L"], np.nan]
final_table["AnswerRecall"] = [generation_metrics(final_med, mt["answers"].tolist())["AnswerRecall"], np.nan]
final_table.to_csv(RESULT_DIR / "final_pipeline.csv")
print(f"THRESHOLD = {THRESHOLD:.4f}")
final_table

### 5-1. 결과 저장과 사례 확인

In [ ]:
cases = mt[["query_id", "query", "answers"]].copy()
for cfg, p in preds.items():
    cases[cfg] = p
cases["Final"] = final_med
cases["top_score"] = top_scores("med_test")
cases.to_json(RESULT_DIR / "med_test_predictions.jsonl", orient="records", lines=True, force_ascii=False)

ood_cases = qsets["ood_test"][["query_id", "query", "answers"]].copy()
ood_cases["Final"] = final_ood
ood_cases["top_score"] = top_scores("ood_test")
ood_cases.to_json(RESULT_DIR / "ood_test_predictions.jsonl", orient="records", lines=True, force_ascii=False)

pd.set_option("display.max_colwidth", 200)
display(cases.sample(5, random_state=SEED)[["query", "answers", "No-Retrieval", "Final"]])
display(ood_cases[~ood_cases["Final"].map(is_refusal)].head(5))   # 거절하지 못한 OOD 사례

## 6. 대화형 테스트

In [ ]:
def ask(query, show_sources=True):
    bm25_ids, _ = bm25_search([query])
    dense_ids, _ = dense_search([query])
    ids, scores = rerank([query], rrf_fuse(bm25_ids, dense_ids))
    ids, scores = ids[0][:K_CTX], scores[0][:K_CTX]
    if scores[0] < THRESHOLD:
        answer = REFUSAL
    else:
        answer = generate([build_messages(query, ids)], batch_size=1)[0]
    print(f"Q: {query}\nA: {answer}  (top score {scores[0]:.3f}, threshold {THRESHOLD:.3f})")
    if show_sources and not is_refusal(answer):
        for i, (pid, s) in enumerate(zip(ids, scores), 1):
            print(f"  [{i}] ({s:.3f}) {corpus.at[pid, 'url']}\n      {passages[pid][:150]}...")

ask("What are the symptoms of iron deficiency anemia?")
ask("Who won the 2010 FIFA World Cup?")